In [ ]:
# HRM analysis of DPYD c.1236G>A (HapB3) from a Rotor-Gene 6000 run file. 2026-10-27 Omid Fotouhi
# A .rex file is plain XML, so the raw melt data can be read without the instrument software.

import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

rex = open("raw_file.rex", encoding="utf-8", errors="replace").read()

In [11]:
# The HRM channel holds one <Reading> per tube: fluorescence at each temperature step.
hrm = [c for c in re.findall(r"<RawChannel>(.*?)</RawChannel>", rex, re.S) if "HRM A.HRM" in c][0]
curves = [np.array(r.split(), dtype=float) for r in re.findall(r"<Reading>(.*?)</Reading>", hrm, re.S)]

start = float(re.search(r"<StartX>(.*?)</StartX>", hrm).group(1))
stepsize = float(re.search(r"<StepX>(.*?)</StepX>", hrm).group(1))
temp = start + stepsize * np.arange(len(curves[0]))

print(len(curves), "tubes,", temp[0], "to", temp[-1], "C")

72 tubes, 70.0 to 85.0 C


In [12]:
# Tube numbers from the run sheet. Positions 37-72 were empty.
wild_type = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 33, 34, 35]
heterozygous = [12, 19, 21, 22, 27, 28]
gg_oligo = [29, 30, 31]
ga_oligo = [23, 25, 26]
aa_oligo = [14, 17, 20]

In [13]:
# QC: the raw fluorescence range tells you how much product a tube made.
# Below ~25 units there is too little signal to call a genotype.
amplitude = np.array([c.max() - c.min() for c in curves[:36]])
print("tubes failing QC:", [i + 1 for i in range(36) if amplitude[i] < 25])


def normalise(curve):
    # Set the flat baseline before melting to 100% and after melting to 0%,
    # so curves differing only in DNA amount become comparable.
    before = curve[(temp >= 74.0) & (temp <= 76.0)].mean()
    after = curve[(temp >= 82.5) & (temp <= 84.5)].mean()
    return (curve - after) / (before - after) * 100

tubes failing QC: [24, 32, 36]


In [14]:
def derivative(curve):
    # Savitzky-Golay over 21 points (1.0 C) smooths and differentiates in one pass.
    return -savgol_filter(curve, 21, 3, deriv=1, delta=stepsize)

In [15]:
def tm(curve):
    # Melting temperature is the peak of -dF/dT inside the transition.
    melt = (temp > 76) & (temp < 83)
    return temp[melt][np.argmax(derivative(curve)[melt])]


for name, tubes in [("GG clinical", wild_type), ("GA clinical", heterozygous),
                    ("GG oligo", gg_oligo), ("GA oligo", ga_oligo), ("AA oligo", aa_oligo)]:
    values = [tm(curves[t - 1]) for t in tubes]
    print(f"{name:<12} n={len(tubes):2d}  Tm {np.mean(values):.2f} +/- {np.std(values):.2f} C")
    assert all(amplitude[t - 1] >= 25 for t in tubes), f"{name} contains a tube below QC"

colour = {"GG": "#c06000", "GA": "#1f6fb4", "AA": "#7b3fa0"}
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "Arial"

GG clinical  n=15  Tm 79.80 +/- 0.17 C
GA clinical  n= 6  Tm 80.13 +/- 0.04 C
GG oligo     n= 3  Tm 79.85 +/- 0.07 C
GA oligo     n= 3  Tm 80.03 +/- 0.05 C
AA oligo     n= 3  Tm 79.40 +/- 0.00 C


In [16]:

def make_figure(groups, reference_tubes, filename):
    # Three panels: normalised melt, derivative, and difference against the reference genotype.
    reference = np.mean([normalise(curves[t - 1]) for t in reference_tubes], axis=0)
    fig, axes = plt.subplots(3, 1, figsize=(7.09, 8.86))

    for label, tubes, key in groups:
        for t in tubes:
            raw = curves[t - 1]
            axes[0].plot(temp, normalise(raw), color=colour[key], lw=1.0)
            axes[1].plot(temp, derivative(raw), color=colour[key], lw=1.0)
            axes[2].plot(temp, normalise(raw) - reference, color=colour[key], lw=1.0)

    for ax in axes:
        ax.set_xlim(77, 82)
        ax.grid(True, color="#dcdcdc", lw=0.5)
        ax.set_axisbelow(True)
        ax.set_xlabel("Temperature (°C)")

    axes[0].set_ylabel("Normalised fluorescence (%)")
    axes[1].set_ylabel("−dF/dT")
    axes[2].set_ylabel("Difference vs reference")
    axes[0].legend([plt.Line2D([], [], color=colour[k], lw=2) for _, _, k in groups],
                   [label for label, _, _ in groups], frameon=False, loc="lower left")

    fig.tight_layout()
    fig.savefig(filename)
    print("wrote", filename)


make_figure([("Wild type GG", gg_oligo, "GG"),
             ("Heterozygous GA", ga_oligo, "GA"),
             ("Homozygous variant AA", aa_oligo, "AA")],
            gg_oligo, "Figure3_synthetic_controls.svg")

make_figure([("Wild type GG, clinical", wild_type, "GG"),
             ("Heterozygous GA, clinical", heterozygous, "GA"),
             ("Homozygous variant AA, synthetic control", aa_oligo, "AA")],
            wild_type, "Figure4_clinical_samples.svg")


wrote Figure3_synthetic_controls.svg
wrote Figure4_clinical_samples.svg
